In [2]:
import os
import numpy as np
import rasterio
from rasterio.enums import Resampling
import imageio
import rioxarray as rxr
from tqdm import tqdm
import xarray as xr
import pandas as pd

import matplotlib.pyplot as plt
import geopandas as gpd


## test avec uniquement image raster 

In [ ]:
# Define the input and output directories
input_dir = "H:/data/airparif/pic_pollution/PM10_2017_pic"
output_dir = "H:/data/airparif/pic_pollution/PM10_2017_pic/output"
os.makedirs(output_dir, exist_ok=True)

def process_raster(file,file_path, output_path):
    ds = rxr.open_rasterio(file_path, masked=True)
    data = ds.values[0]

    binary_mask = np.where(data > 50, 255, 0).astype(np.uint8)

    # Create a new DataArray with the binary mask
    binary_mask_da = xr.DataArray(binary_mask, coords={"y": ds.coords["y"], "x": ds.coords["x"]}, dims=("y", "x"), attrs=ds.attrs)

    binary_mask_da.rio.to_raster(os.path.join(output_dir,f"{file[:-3]}.tif"), compress='DEFLATE', zlevel=9, tiled=True)

# Process all raster files in the input directory
for file in tqdm(os.listdir(input_dir)):
    if file.endswith('.nc'):
        input_path = os.path.join(input_dir, file)
        output_path = os.path.join(output_dir, file)
        process_raster(file,input_path, output_path)




In [27]:
output_files = [os.path.join(output_dir, f) for f in sorted(os.listdir(output_dir)) if f.endswith('.tif')]

images = []
for output_file in tqdm(output_files):
    # ds = rxr.open_rasterio(output_file, masked=True)
    # image = ds.values[0]
    images.append(imageio.imread(output_file))

# Save the animation as a gif
imageio.mimsave('PM10_pic_pollution.gif', images, duration=5)

  0%|          | 0/365 [00:00<?, ?it/s]C:\Users\jbocque1\AppData\Local\Temp\ipykernel_22992\1753111811.py:7: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images.append(imageio.imread(output_file))
100%|██████████| 365/365 [00:14<00:00, 25.24it/s]


## Mise en forme données patients afin de les afficher en parallèle

In [3]:
df = pd.read_csv("R:/Direction_Data/0_Projets/Projet_CANCAIR/data/data_cleaned/patients_FR_IDF_geocoded_adulte_clinique.csv",sep=";")
df_EM = pd.read_csv("R:/Direction_Data/0_Projets/Projet_CANCAIR/data/data_clinique/medical_events_all.csv")

In [4]:
id_patients = df['pseudo_provisoire'].tolist()
df_EM_id = df_EM[df_EM['pseudo_provisoire'].isin(id_patients)]

df_EM_geodata = pd.merge(df_EM_id,df[['pseudo_provisoire','x','y']], on='pseudo_provisoire',how="left")

def parse_dates(date_str):
    try:
        return pd.to_datetime(date_str, format='%Y-%m-%d %H:%M:%S')
    except ValueError:
        return pd.to_datetime(date_str, format='%d/%m/%Y')

df_EM_geodata['standardized_dates'] = df_EM_geodata['Date_debut'].apply(parse_dates)

df_EM_geodata['standardized_dates'] = df_EM_geodata['standardized_dates'].dt.strftime('%Y-%m-%d')

In [13]:
input_dir = "C:/Users/jbocque1/Desktop/projet/data/PM10_2017_pic"
output_dir = "C:/Users/jbocque1/Desktop/projet/data/PM10_2017_pic/output_s50"
output_plot = "C:/Users/jbocque1/Desktop/projet/data/PM10_2017_pic/output_s50/plot"
idf_plot = (gpd.read_file("H:/data/zones_geographiques/departements/IDF/dpt_idf.shp").set_crs(epsg=2154)).to_crs(epsg=27572)

os.makedirs(output_dir, exist_ok=True)

gdf = (gpd.GeoDataFrame(df_EM_geodata, geometry=gpd.points_from_xy(df_EM_geodata.x, df_EM_geodata.y)).set_crs(epsg=4326)).to_crs(epsg=27572)

def process_raster_points(input_path, output_path, file, events,date):#file,file_path, ):
    ds = rxr.open_rasterio(input_path, masked=True)
    data = ds.values[0]

    binary_mask = np.where(data > 25, 255, 0).astype(np.uint8)

    # Create a new DataArray with the binary mask
    binary_mask_da = xr.DataArray(binary_mask, coords={"y": ds.coords["y"], "x": ds.coords["x"]}, dims=("y", "x"), attrs=ds.attrs)

    files_done = os.listdir(output_dir)
    file_name = os.path.join(output_dir,f"{file[:-3]}.tif")
                             
    if file_name not in files_done :
        binary_mask_da.rio.to_raster(file_name, compress='DEFLATE', zlevel=9, tiled=True)


    # Overlay patient events
    fig, ax = plt.subplots(figsize=(10, 10))

    ax.imshow(binary_mask, cmap='gray', interpolation='nearest', extent=[ds.coords["x"].min(), ds.coords["x"].max(), ds.coords["y"].min(), ds.coords["y"].max()])
    idf_plot.plot(ax=ax, edgecolor='crimson', linewidth=2, zorder=4,facecolor='none')

    # Plot patient events
    if not events.empty:
        ax.scatter(x=events.geometry.x, y=events.geometry.y, color='dodgerblue', s=100, edgecolor='black', zorder=5)

    ax.text(0.05, 0.95, date, transform=ax.transAxes, fontsize=14, verticalalignment='top', bbox=dict(facecolor='white', alpha=0.5))

    output_path = os.path.join(output_plot,f"{file[:-3]}.png")

    fig.savefig(output_path)
    plt.close()


for file in tqdm(os.listdir(input_dir)):
    if file.endswith('.nc'):
        input_path = os.path.join(input_dir, file)
        output_path = os.path.join(output_dir, file)

        pollDate = file.split('_')[-1].split('.')[0]
        date =  pollDate[:4] +'-' + pollDate[4:6] +'-' + pollDate[6:]

        events_on_date = gdf[gdf['standardized_dates'] == date]
        process_raster_points(input_path, output_path,file, events_on_date,date)

output_files = [os.path.join(output_plot, f) for f in sorted(os.listdir(output_plot)) if f.endswith('.png')]
images = []
for output_file in tqdm(output_files):
    images.append(imageio.imread(output_file))

imageio.mimsave('PM10_pic_pollution_EM_s50.gif', images, duration=5)

  0%|          | 0/365 [00:00<?, ?it/s]C:\Users\jbocque1\AppData\Local\Temp\ipykernel_23368\2179578118.py:58: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images.append(imageio.imread(output_file))
100%|██████████| 365/365 [00:05<00:00, 63.72it/s]


In [12]:
output_files = [os.path.join(output_plot, f) for f in sorted(os.listdir(output_plot)) if f.endswith('.png')]
images = []
for output_file in tqdm(output_files):
    images.append(imageio.imread(output_file))

imageio.mimsave('PM10_pic_pollution_EM_s25.gif', images, duration=5)

  0%|          | 0/365 [00:00<?, ?it/s]C:\Users\jbocque1\AppData\Local\Temp\ipykernel_23368\3100464360.py:4: DeprecationWarning: Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.
  images.append(imageio.imread(output_file))
100%|██████████| 365/365 [00:05<00:00, 69.58it/s]
